In [ ]:
# ============================================
# RUN THIS FIRST — Colab Setup
# ============================================
!pip install datasets feedparser sentence-transformers -q

# Clone the repo to access project structure
!git clone https://github.com/melissanoarianna1-commits/finpulse-ai.git 2>/dev/null || echo 'Repo already cloned'
import os
os.chdir('/content/finpulse-ai')
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/validated', exist_ok=True)
os.makedirs('models/sentiment', exist_ok=True)
os.makedirs('models/recommender', exist_ok=True)
os.makedirs('docs', exist_ok=True)
print('✅ Setup complete. Working directory:', os.getcwd())

# Model 2: Paper Relevance Recommender

## Overview
Builds a **paper relevance scoring model** using **Sentence-BERT embeddings** and
**logistic regression**. Given a paper abstract, predicts relevance to Banking & Finance
and outputs a score (0-100%).

## Role in FinPulse AI
Powers the **relevance percentage bars** on paper cards and determines ranking in the daily feed.

## Connection to Financial Risk
Same architecture as recommendation/ranking systems used in:
- Portfolio optimization (ranking assets by expected utility)
- Credit scoring (ranking applicants by risk profile)
- Alert prioritization (ranking risk events to investigate)

## Approach
1. Fetch paper abstracts from arXiv (finance vs. non-finance)
2. Generate Sentence-BERT embeddings
3. Train logistic regression classifier
4. Evaluate on validation set

## 1. Collect Paper Abstracts

Naturally labeled dataset:
- **Relevant (1)**: arXiv q-fin (Risk Management, General Finance, Computational Finance)
- **Not relevant (0)**: arXiv cs.CV, physics.bio-ph, math.CO

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import feedparser
import time
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

def fetch_arxiv(query, max_results=200):
    url = f'http://export.arxiv.org/api/query?search_query={query}&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending'
    feed = feedparser.parse(url)
    papers = []
    for entry in feed.entries:
        abstract = entry.summary.replace('\n', ' ').strip()
        if len(abstract) > 50:
            papers.append({
                'title': entry.title.replace('\n', ' ').strip(),
                'abstract': abstract,
                'authors': ', '.join([a.name for a in entry.authors]),
                'date': entry.published[:10],
                'url': entry.link,
            })
    return pd.DataFrame(papers)

print('Fetching finance papers...')
df_fin = fetch_arxiv('cat:q-fin.RM+OR+cat:q-fin.GN+OR+cat:q-fin.CP', 200)
df_fin['relevant'] = 1
print(f'  → {len(df_fin)} papers')

time.sleep(3)

print('Fetching non-finance papers...')
df_other = fetch_arxiv('cat:cs.CV+OR+cat:physics.bio-ph+OR+cat:math.CO', 200)
df_other['relevant'] = 0
print(f'  → {len(df_other)} papers')

df = pd.concat([df_fin, df_other], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nTotal: {len(df)} ({df["relevant"].sum()} relevant, {(df["relevant"]==0).sum()} not relevant)')

## 2. Train/Validation Split

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['relevant'])
print(f'Training: {len(train_df)} | Validation: {len(val_df)}')

## 3. Generate Sentence-BERT Embeddings

**Sentence-BERT** converts each abstract into a 384-dimensional dense vector.
Papers with similar content get similar vectors. We use `all-MiniLM-L6-v2` — fast and high quality.

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Embedding dim: {encoder.get_sentence_embedding_dimension()}')

print('Encoding training abstracts...')
train_emb = encoder.encode(train_df['abstract'].tolist(), show_progress_bar=True, batch_size=32)

print('Encoding validation abstracts...')
val_emb = encoder.encode(val_df['abstract'].tolist(), show_progress_bar=True, batch_size=32)

print(f'\nTrain shape: {train_emb.shape} | Val shape: {val_emb.shape}')

## 4. Train Classifier

**Logistic regression** on top of embeddings because:
- Outputs probabilities → 0-100% relevance scores
- Interpretable and fast
- Simple classifier avoids overfitting on small data

In [ ]:
clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(train_emb, train_df['relevant'].values)

val_preds = clf.predict(val_emb)
val_probs = clf.predict_proba(val_emb)[:, 1]
print('✅ Training complete')

## 5. Evaluate on Validation Set

In [ ]:
print('='*60)
print('CLASSIFICATION REPORT — Validation Set')
print('='*60)
print(classification_report(val_df['relevant'].values, val_preds, target_names=['Not Relevant', 'Relevant']))

auc = roc_auc_score(val_df['relevant'].values, val_probs)
print(f'AUC-ROC: {auc:.4f}')

In [ ]:
# Confusion Matrix + ROC Curve
cm = confusion_matrix(val_df['relevant'].values, val_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Relevant', 'Relevant'],
            yticklabels=['Not Relevant', 'Relevant'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix', fontweight='bold')

fpr, tpr, _ = roc_curve(val_df['relevant'].values, val_probs)
axes[1].plot(fpr, tpr, color='#3b82f6', linewidth=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('docs/recommender_evaluation.png', dpi=150)
plt.show()

In [ ]:
# Show sample relevance scores
print('Sample Relevance Scores:')
print('='*80)
for i in range(min(8, len(val_df))):
    row = val_df.iloc[i]
    score = val_probs[i] * 100
    actual = 'RELEVANT' if row['relevant'] == 1 else 'NOT RELEVANT'
    print(f'\n  [{actual}] Score: {score:.1f}%')
    print(f'  {row["title"][:75]}...')

## 6. Save Model

In [ ]:
import os
os.makedirs('models/recommender', exist_ok=True)

with open('models/recommender/classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

with open('models/recommender/config.txt', 'w') as f:
    f.write(f'encoder=all-MiniLM-L6-v2\n')
    f.write(f'auc={auc:.4f}\n')
    f.write(f'n_train={len(train_df)}\n')
    f.write(f'n_val={len(val_df)}\n')

print('✅ Model saved to models/recommender/')
print('\n--- Model 2 Complete ---')
print('Both models trained and validated!')